In [1]:
# 标准化
pr1=PolynomialFeatures(degree=2)

# powers_ =（powers_ 是"列清单"的转置视图——每行是一个列的配方，所以 11 个新增列在）
# [[0 0 0 0]   ①  1（截距列，全1）          [新增]
#  [1 0 0 0]   ②  x1 = horsepower          （原有）
#  [0 1 0 0]   ③  x2 = curb-weight         （原有）
#  [0 0 1 0]   ④  x3 = engine-size         （原有）
#  [0 0 0 1]   ⑤  x4 = highway-mpg         （原有）
#  [2 0 0 0]   ⑥  x1² = horsepower²        [新增]
#  [1 1 0 0]   ⑦  x1·x2                    [新增]
#  [1 0 1 0]   ⑧  x1·x3                    [新增]
#  [1 0 0 1]   ⑨  x1·x4                    [新增]
#  [0 2 0 0]   ⑩  x2²                      [新增]
#  [0 1 1 0]   ⑪  x2·x3                    [新增]
#  [0 1 0 1]   ⑫  x2·x4                    [新增]
#  [0 0 2 0]   ⑬  x3²                      [新增]
#  [0 0 1 1]   ⑭  x3·x4                    [新增]
#  [0 0 0 2]]  ⑮  x4²                      [新增]
# 升维
x_train_pr1=pr1.fit_transform(x_train[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

x_test_pr1=pr1.fit_transform(x_test[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']])

x_train_pr1.shape

# 注意！！！
# 原始数据集如下：
# horsepower	curb-weight	engine-size	highway-mpg	normalized-losses	symboling
# 111	2500	134	21	122	1
# 154	2880	160	19	137	0
# ...	...	...	...	...	...

# 升维后的数据集如下：
# 1	horsepower	curb-weight	...	horsepower²	horsepower×curb-weight	...
# 1	111	2500	...	12321	277500	...
# 1	154	2880	...	23716	443520	...
# ...	...	...	...	...	...	...

# 建立模型、算出系数、里面传入升维后的训练集和价格训练集
poly1=LinearRegression().fit(x_train_pr1,y_train)

# 预测后，输出测试集的密度函数
yhat_test1=poly1.predict(x_test_pr1)

Title='Distribution  Plot of  Predicted Value Using Test Data vs Data Distribution of Test Data'

DistributionPlot(y_test, yhat_test1, "Actual Values (Test)", "Predicted Values (Test)", Title)

# Ridge 回归是什么
# 高维多项式特征容易导致过拟合（前面 Question 4 已经演示了）
# 引入 Ridge 回归，通过 L2 正则化压制系数大小，缓解过拟合
# alpha 控制正则化强度 alpha的值一般再0-10000之间
# 能看到效果	alpha=1 已经有正则化效果，可以和前面的普通多项式回归做对比
# 不是最终答案	后续遍历 0~9990 才是真正调参的过程，alpha=1 只是"先跑一个看看"
# 情况	类比	模型表现
# 欠拟合	书都没看，直接去考	训练集和测试集都差
# 刚好	学会了 general 的解题方法	训练集和测试集都好
# 过拟合	把练习题的答案死记硬背下来了	训练集满分，换个题就不会了
# 为什么会过拟合？
# 模型太复杂：degree=5 的多项式有 6 个参数去拟合 1 个特征，曲线可以扭来扭去穿过每一个训练点
# 数据太少：样本量不够，模型容易把个别噪声点当成规律
# 特征太多：多项式展开后特征维度暴增（4 维 → 15 维），自由度太高
from sklearn.linear_model import Ridge
#ridge括号中可以不写，默认alpha为1
RigeModel=Ridge(alpha=1)
RigeModel.fit(x_train_pr, y_train)
yhat = RigeModel.predict(x_test_pr)

# .values 的作用就是把 pandas Series → numpy 数组，去掉索引和名称等元信息，只保留数据本身。纯粹是为了显示整洁，不影响实际数值
print('predicted:', yhat[0:4])
print('test set :', y_test[0:4].values)

# 导入 tqdm 库，用于显示进度条，让长时间循环可视化。
from tqdm import tqdm

Rsqu_test = [] #存测试集的R平方
Rsqu_train = [] #存训练集的R平方
dummy1 = [] #其余变量
# 生成 alpha 候选值数组：[0, 10, 20, 30, ..., 9990]，共 1000 个值。range(0,1000) 生成 0~999，乘以 10 变成步长为 10 的等差序列。
#也可以用arange(0,1000,10)
Alpha = 10 * np.array(range(0,1000))
pbar = tqdm(Alpha) #把 Alpha 数组包装成一个带进度条的可迭代对象。

for alpha in pbar:  #循环的次数与alpha的总数有关  把不同alpha值下面的函数所对应的R值（训练集和测试集的R平方）全部存起来
    RigeModel = Ridge(alpha=alpha)
    RigeModel.fit(x_train_pr, y_train)
    test_score, train_score = RigeModel.score(x_test_pr, y_test), RigeModel.score(x_train_pr, y_train)
    # 在进度条右侧实时显示当前的 Test Score 和 Train Score，方便观察变化趋势。
    # 45%|████▌     | 450/1000 [00:03<00:04, Test Score: 0.72, Train Score: 0.78]
    pbar.set_postfix({"Test Score": test_score, "Train Score": train_score})

    #把结果分别存成两个数组
    Rsqu_test.append(test_score)
    Rsqu_train.append(train_score)

width = 12
height = 10
plt.figure(figsize=(width, height))

# 默认画蓝线 ：横轴是 alpha，纵轴是测试集R平方
plt.plot(Alpha,Rsqu_test, label='validation data  ')
#横轴alpha，纵轴是训练集R平方
plt.plot(Alpha,Rsqu_train, 'r', label='training Data ')
plt.xlabel('alpha')
plt.ylabel('R^2')
plt.legend()



NameError: name 'PolynomialFeatures' is not defined